In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['figure.figsize'] = 12,6
plt.rcParams['font.size'] = 14
plt.rcParams['axes.unicode_minus'] = False

# 데이터 전처리 관련 ####################################################
# 결측치 처리
from sklearn.impute import SimpleImputer
# 표준화
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import MinMaxScaler
# 인코더
from sklearn.preprocessing import LabelEncoder
# 원핫 인코더
from sklearn.preprocessing import OneHotEncoder

# 학습 모델 성능 관련 ####################################################
# 원하는 비율로 데이터를 나누기 위해
from sklearn.model_selection import train_test_split
# K-Fold 교차 검증
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import KFold
from sklearn.model_selection import StratifiedKFold
# 학습곡선
from sklearn.model_selection import learning_curve
# 하이퍼 파라미터 튜닝
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint
import optuna
from optuna.samplers import TPESampler # 데이터를 랜덤샘플링하기 위함
from optuna.pruners import MedianPruner
optuna.logging.set_verbosity(optuna.logging.WARNING) # 수치가 떨어지면 경고로그가 뜨는데 그거를 막아줌

# 모델 성능평가 #############################################
# 회귀용
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import mean_squared_error
from sklearn.metrics import root_mean_squared_error
from sklearn.metrics import r2_score
# 분류용
from sklearn.metrics import confusion_matrix
from sklearn.metrics import accuracy_score
from sklearn.metrics import precision_score
from sklearn.metrics import recall_score
from sklearn.metrics import f1_score

from sklearn.metrics import roc_curve
from sklearn.metrics import auc
from sklearn.metrics import roc_auc_score
from sklearn.metrics import ConfusionMatrixDisplay

# 피처 선택 ################################################
from sklearn.feature_selection import VarianceThreshold
from sklearn.feature_selection import RFE
from sklearn.inspection import permutation_importance

# 학습모델 ##################################################
#분류
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import GradientBoostingClassifier
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
from sklearn.naive_bayes import GaussianNB
from catboost import CatBoostClassifier

#회귀
from sklearn.neighbors import KNeighborsRegressor
from sklearn.linear_model import LinearRegression
from sklearn.linear_model import Ridge
from sklearn.linear_model import Lasso
from sklearn.linear_model import ElasticNet
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import GradientBoostingRegressor
from lightgbm import LGBMRegressor
from xgboost import XGBRegressor
from sklearn.linear_model import BayesianRidge
from catboost import CatBoostRegressor

# 결정트리를 시각화할 수 있는 라이브러리
from sklearn.tree import plot_tree

# 차원축소
from sklearn.decomposition import PCA
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.manifold import TSNE

# 연관규칙 학습
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori
from mlxtend.frequent_patterns import fpgrowth
from mlxtend.frequent_patterns import association_rules

# 군집
from sklearn.cluster import KMeans
from sklearn.cluster import DBSCAN
from scipy.cluster.hierarchy import dendrogram, linkage, fcluster
from sklearn.mixture import GaussianMixture
from sklearn.cluster import MeanShift, estimate_bandwidth

# 파이프라인
from sklearn.pipeline import Pipeline

# KDE를 그리기 위한 통계값을 구할 수 있는 함수
from scipy.stats import gaussian_kde

# 피어슨 상관 계수 (연속형 수치형 데이터 vs 연속형 수치형 데이터)
from scipy.stats import pearsonr
# 카이제곱 검증 (범주형 데이터 vs 범주현 데이터, 순위 x)
from scipy.stats import chi2_contingency
# 스피어만 상관계수 (범주형 데이터 vs 범주형 데이터, 순위 O)
from scipy.stats import spearmanr
# 포인트 이분 상관계수 (범주형 데이터 vs 연속형 수치형 데이터)
from scipy.stats import pointbiserialr

# 객체를 파일에 저장
import pickle

# 불필요한 경고 뜨지 않게
import warnings
warnings.filterwarnings('ignore')

In [2]:
test_df = pd.read_csv('data/bike_sharing_test5.csv')
test_df

,temp,humidity,year,log_windspeed,season_1,season_2,season_3,season_4,weather_1,weather_3,...,hour_14,hour_15,hour_16,hour_17,hour_18,hour_19,hour_20,hour_21,hour_22,hour_23
0,10.66,56,1,3.295937,1,0,0,0,1,0,...,0,0,0,0,0,0,0,0,0,0
1,10.66,56,1,0.000000,1,0,0,0,1,0,...,0,0,0,0,0,0,0,0,0,0
2,10.66,56,1,0.000000,1,0,0,0,1,0,...,0,0,0,0,0,0,0,0,0,0
3,10.66,56,1,2.485023,1,0,0,0,1,0,...,0,0,0,0,0,0,0,0,0,0
4,10.66,56,1,2.485023,1,0,0,0,1,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6488,10.66,60,0,2.485023,1,0,0,0,0,0,...,0,0,0,0,0,1,0,0,0,0
6489,10.66,60,0,2.485023,1,0,0,0,0,0,...,0,0,0,0,0,0,1,0,0,0
6490,10.66,60,0,2.485023,1,0,0,0,1,0,...,0,0,0,0,0,0,0,1,0,0
6491,10.66,56,0,2.302395,1,0,0,0,1,0,...,0,0,0,0,0,0,0,0,1,0


### 예측 준비

In [3]:
# 저장된 모델을 불러온다.
with open('model/project2.dat', 'rb') as fp :
    model = pickle.load(fp)
    scaler = pickle.load(fp)

In [4]:
display(model)
display(scaler)

StandardScaler()

### 예측

In [5]:
# 표준화
X_scaled = scaler.transform(test_df)

In [7]:
# 예측
y_pred = model.predict(X_scaled)
y_pred

array([2.54446258, 2.54140151, 2.55027643, ..., 4.72949142, 4.46010674,
       3.96673357], shape=(6493,))

In [8]:
# scale 된거니 복원해야함
y_original = np.expm1(y_pred)
y_original

array([ 11.73638143,  11.69745407,  11.81064449, ..., 112.23795749,
        85.49674149,  51.8117428 ], shape=(6493,))

In [9]:
test_df['count'] = y_original
test_df

,temp,humidity,year,log_windspeed,season_1,season_2,season_3,season_4,weather_1,weather_3,...,hour_15,hour_16,hour_17,hour_18,hour_19,hour_20,hour_21,hour_22,hour_23,count
0,10.66,56,1,3.295937,1,0,0,0,1,0,...,0,0,0,0,0,0,0,0,0,11.736381
1,10.66,56,1,0.000000,1,0,0,0,1,0,...,0,0,0,0,0,0,0,0,0,11.697454
2,10.66,56,1,0.000000,1,0,0,0,1,0,...,0,0,0,0,0,0,0,0,0,11.810644
3,10.66,56,1,2.485023,1,0,0,0,1,0,...,0,0,0,0,0,0,0,0,0,6.263675
4,10.66,56,1,2.485023,1,0,0,0,1,0,...,0,0,0,0,0,0,0,0,0,1.585112
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6488,10.66,60,0,2.485023,1,0,0,0,0,0,...,0,0,0,0,1,0,0,0,0,203.445331
6489,10.66,60,0,2.485023,1,0,0,0,0,0,...,0,0,0,0,0,1,0,0,0,143.332909
6490,10.66,60,0,2.485023,1,0,0,0,1,0,...,0,0,0,0,0,0,1,0,0,112.237957
6491,10.66,56,0,2.302395,1,0,0,0,1,0,...,0,0,0,0,0,0,0,1,0,85.496741


In [10]:
# 최종저장
test_df.to_csv('data/bike_sharing_result.csv', index=False)